In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from aeon.classification.convolution_based import (
    MultiRocketHydraClassifier,
    RocketClassifier,
    MultiRocketClassifier,
    MiniRocketClassifier,
)
from aeon.classification.feature_based import Catch22Classifier
from aeon.classification.interval_based import QUANTClassifier
from aeon.classification.dummy import DummyClassifier
from tscglue.data_loader import load_fold

In [ ]:
# datasets spanning different sizes and series lengths
from tscglue.models import TSCGlue


DATASETS = [
    # tiny
    "Chinatown",             # (20 train,   345 test, len=24)
    "ItalyPowerDemand",      # (67 train,  1029 test, len=24)
    "SonyAIBORobotSurface1", # (20 train,   601 test, len=70)
    "TwoLeadECG",            # (23 train,  1139 test, len=82)
    "MoteStrain",            # (20 train,  1252 test, len=84)
    "GunPoint",              # (50 train,   150 test, len=150)
    "Coffee",                # (28 train,    28 test, len=286)
    # small
    "ECG200",                # (100 train,  100 test, len=96)
    "ArrowHead",             # (36 train,   175 test, len=251)
    "Trace",                 # (100 train,  100 test, len=275)
    "Adiac",                 # (390 train,  391 test, len=176)
    # medium
    "CricketX",              # (390 train,  390 test, len=300)
    "FacesUCR",              # (200 train, 2050 test, len=131)
]

MODELS = {
    # n_jobs=1
    "catch22-j1":      lambda: Catch22Classifier(n_jobs=1),
    "rocket-j1":       lambda: RocketClassifier(n_jobs=1),
    "minirocket-j1":   lambda: MiniRocketClassifier(n_jobs=1),
    "multirocket-j1":  lambda: MultiRocketClassifier(n_jobs=1),
    "hydra-j1":        lambda: MultiRocketHydraClassifier(n_jobs=1),
    "tscglue-j1":      lambda: TSCGlue(n_jobs=1),
    # n_jobs=4
    "dummy":           lambda: DummyClassifier(),
    "catch22":         lambda: Catch22Classifier(n_jobs=4),
    "quant":           lambda: QUANTClassifier(),
    "rocket":          lambda: RocketClassifier(n_jobs=4),
    "minirocket":      lambda: MiniRocketClassifier(n_jobs=4),
    "multirocket":     lambda: MultiRocketClassifier(n_jobs=4),
    "hydra":           lambda: MultiRocketHydraClassifier(n_jobs=4),
    "tscglue":         lambda: TSCGlue(n_jobs=4),
    # n_jobs=12
    "catch22-j12":     lambda: Catch22Classifier(n_jobs=12),
    "rocket-j12":      lambda: RocketClassifier(n_jobs=12),
    "minirocket-j12":  lambda: MiniRocketClassifier(n_jobs=12),
    "multirocket-j12": lambda: MultiRocketClassifier(n_jobs=12),
    "hydra-j12":       lambda: MultiRocketHydraClassifier(n_jobs=12),
    "tscglue-j12":     lambda: TSCGlue(n_jobs=12),
}

RESULTS_DIR = "../results/timing_benchmark"

In [ ]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

def result_path(dataset, model_name):
    return os.path.join(RESULTS_DIR, f"{dataset}__{model_name}.csv")

def load_cached(dataset, model_name):
    p = result_path(dataset, model_name)
    if os.path.exists(p):
        return pd.read_csv(p).to_dict("records")[0]
    return None

def save_result(row):
    pd.DataFrame([row]).to_csv(result_path(row["dataset"], row["model"]), index=False)

results = []

for dataset in DATASETS:
    print(f"\n=== {dataset} ===")
    X_train, y_train, X_test, y_test = None, None, None, None  # load lazily

    for model_name, model_fn in MODELS.items():
        cached = load_cached(dataset, model_name)
        if cached is not None:
            print(f"  {model_name:12s}  [cached]  fit={cached['fit_time']:.2f}s  acc={cached['accuracy']:.4f}")
            results.append(cached)
            continue

        if X_train is None:
            X_train, y_train, X_test, y_test = load_fold(dataset, fold=0)
            print(f"  train={X_train.shape}, test={X_test.shape}")

        model = model_fn()

        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        fit_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        y_pred = model.predict(X_test)
        pred_time = time.perf_counter() - t0

        acc = accuracy_score(y_test, y_pred)
        print(f"  {model_name:12s}  fit={fit_time:.2f}s  pred={pred_time:.2f}s  acc={acc:.4f}")

        row = {
            "dataset":    dataset,
            "model":      model_name,
            "n_train":    X_train.shape[0],
            "n_test":     X_test.shape[0],
            "series_len": X_train.shape[2],
            "fit_time":   fit_time,
            "pred_time":  pred_time,
            "accuracy":   acc,
        }
        save_result(row)
        results.append(row)

df = pd.DataFrame(results)
print("\nDone.")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gmean

#BASELINE = "catch22"
BASELINE = "dummy"

# accuracy rank per dataset (lower = better)
df["acc_rank"] = df.groupby("dataset")["accuracy"].rank(ascending=False)

pivot = df.pivot(index="dataset", columns="model", values="fit_time")
models = df["model"].unique()

gmean_ratio = {}
mean_rank   = {}
for m in models:
    ratios = pivot[m] / pivot[BASELINE]
    gmean_ratio[m] = gmean(ratios)
    mean_rank[m]   = df[df["model"] == m]["acc_rank"].mean()

fig, ax = plt.subplots(figsize=(7, 5))
for m in models:
    if m == 'dummy':
        continue
    ax.scatter(gmean_ratio[m], mean_rank[m], s=100, zorder=3)
    ax.annotate(m, (gmean_ratio[m], mean_rank[m]),
                textcoords="offset points", xytext=(8, 4), fontsize=11)

#ax.set_xscale("log")
ax.set_xlabel(f"Geometric mean train time ratio (vs {BASELINE})", fontsize=12)
ax.set_ylabel("Mean accuracy rank (lower = better)", fontsize=12)
ax.set_title("Accuracy rank vs train time", fontsize=13)
ax.invert_yaxis()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np

MODELS_PLOT = ["catch22", "rocket", "tscglue-j1", "tscglue", "tscglue-j12"]

pivot_log = (
    df[df["model"].isin(MODELS_PLOT)]
    .groupby(["dataset", "model"])["fit_time"]
    .mean()
    .unstack("model")
    [MODELS_PLOT]
    .apply(np.log10)
)

global_min = pivot_log.min().min()
global_max = pivot_log.max().max()

# Color by tscglue-j1 rank so colormap spreads evenly across datasets
color_vals = pivot_log["tscglue-j1"].rank()

# Shared tick positions and labels
shared_ticks = [-2, -1, 0, 1, 2, 3]
shared_labels = ["0.01s", "0.1s", "1s", "10s", "100s", "1000s"]

dimensions = [
    dict(
        label=col,
        values=pivot_log[col],
        range=[global_min, global_max],
        tickvals=shared_ticks,
        ticktext=shared_labels,
    )
    for col in MODELS_PLOT
]

fig = go.Figure(go.Parcoords(
    line=dict(
        color=color_vals,
        colorscale="viridis",
        showscale=True,
        colorbar=dict(
            title="tscglue-j1<br>time rank",
            tickvals=[color_vals.min(), color_vals.max()],
            ticktext=["fast", "slow"],
        ),
    ),
    dimensions=dimensions,
    labelangle=-20,
))

fig.update_layout(
    title=f"Fit time per dataset — all {len(pivot_log)} datasets",
    height=500,
    margin=dict(l=80, r=80, t=60, b=60),
)
fig.show()
